## Load One Sample Document Per Source

In [ ]:
import sys
import os


backend_path = os.path.abspath(os.path.join("..", "backend"))
sys.path.insert(0, backend_path)

from medrag.ingestion.storage import load_articles, load_drugs, load_guideline

# Load one sample from each source
pubmed_sample = load_articles("diabetes", output_dir="../data/raw/pubmed")[0]
openfda_sample = load_drugs("diabetes", output_dir="../data/raw/openfda")[0]
who_sample = load_guideline("hypertension", output_dir="../data/raw/who")

print("=== PubMed Sample ===")
print("Title:", pubmed_sample.title)
print("Abstract length:", len(pubmed_sample.abstract), "chars")
print()

print("=== OpenFDA Sample ===")
print("Brand name:", openfda_sample.brand_name)
print("Indications length:", len(openfda_sample.indications_and_usage), "chars")
print()

print("=== WHO Sample ===")
print("Title:", who_sample.title)
print("Clean text length:", len(who_sample.clean_text), "chars")
print("Num pages:", who_sample.num_pages)

=== PubMed Sample ===
Title: Determinants and promotion strategies for type 1 diabetes screening in children: A qualitative study from a parental perspective.
Abstract length: 1908 chars

=== OpenFDA Sample ===
Brand name: Glimepiride
Indications length: 623 chars

=== WHO Sample ===
Title: Guideline for the pharmacological treatment of hypertension in adults
Clean text length: 137278 chars
Num pages: 61


## Define the Chunk Data Model

In [1]:
import uuid
from pydantic import BaseModel
from typing import Optional, List

class Chunk(BaseModel):
    chunk_id: str 
    point_id: str           
    text: str                  
    raw_text: str    
    source: str                
    topics: List[str]        
    source_id: str        
    chunk_index: int        
    chunk_type: str = "text"          
    metadata: Optional[dict] = None 

    @staticmethod
    def make_point_id(chunk_id: str) -> str:
        """Deterministic UUID5 from a chunk_id, stable across re-runs."""
        return str(uuid.uuid5(uuid.NAMESPACE_URL, chunk_id))

## Technique 1 — Fixed-Size Chunking

In [ ]:
import tiktoken     # tiktoken is OpenAI's tokenizer library --> Its job is to convert text into tokens.
# This is hypertension --> [2028, 374, 58372, 13] --> These numbers are called tokens.

encoding = tiktoken.get_encoding("cl100k_base")  # matches text-embedding-3-small --> cl100k_base --> This is the tokenizer vocabulary --> Think of it as the dictionary that tells the tokenizer how to split text.


def fixed_size_chunk(text: str, chunk_size: int = 300) -> list:
    """
    Split text into chunks of chunk_size tokens each, with no regard for
    sentence/paragraph boundaries - the naive baseline technique.
    """
    tokens = encoding.encode(text)
    chunks = []
    for i in range(0, len(tokens), chunk_size):
        chunk_tokens = tokens[i:i + chunk_size]
        chunk_text = encoding.decode(chunk_tokens)
        chunks.append(chunk_text)
    return chunks


# Test on the WHO sample (where chunking actually matters)
fixed_chunks = fixed_size_chunk(who_sample.clean_text, chunk_size=300)

print(f"Number of chunks: {len(fixed_chunks)}")
print(f"Average chunk length: {sum(len(c) for c in fixed_chunks) / len(fixed_chunks):.0f} chars")
print()
print("=== Sample chunk (middle of document) ===")
print(fixed_chunks[len(fixed_chunks) // 2])

Number of chunks: 110
Average chunk length: 1248 chars

=== Sample chunk (middle of document) ===
 was causal or confounded by age
and other comorbidities associated with HTN, including obesity, diabetes and chronic kidney disease.
Concerns regarding use of angiotensin-converting enzyme inhibitors (ACEis) in these patients were
raised due to identification of angiotensin-converting enzyme 2 (ACE2), the monocarboxypeptidase
that inactivates angiotensin II and thereby counters the activation of the classic renin–angiotensin–
aldosterone system (RAAS), as the functional receptor for the severe acute respiratory syndrome
coronavirus 2 (SARS-CoV-2) (95, 96). The WHO conducted a rapid review of evidence related to the use
ACEis or ARBs in COVID patients which identified 11 observational studies. No studies were found that
were designed to directly assess whether ACEis or ARBs increase the risk of acquiring COVID-19. After
adjustment for confounders, history of ACEi or ARB use was not found t

## Technique 2 — Sentence/Paragraph-Based Chunking

In [33]:
import re

def sentence_based_chunk(text: str, target_tokens: int = 300) -> list:
    """
    Split text into chunks by grouping whole sentences until reaching
    close to target_tokens, never cutting a sentence mid-way.
    """
    # Simple sentence splitter - splits on '. ', '? ', '! ' followed by a capital letter or newline
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)

    chunks = []
    current_chunk = []
    current_tokens = 0

    for sentence in sentences:
        sentence_tokens = len(encoding.encode(sentence))

        if current_tokens + sentence_tokens > target_tokens and current_chunk:  # 0 + sentence_tokens > target_tokens and current_chunk is not empty = 0 + 12 = 12 > 300 and current_chunk is not empty = False`
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_tokens = 0

        current_chunk.append(sentence)
        current_tokens += sentence_tokens

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


sentence_chunks = sentence_based_chunk(who_sample.clean_text, target_tokens=300)

print(f"Number of chunks: {len(sentence_chunks)}")
print(f"Average chunk length: {sum(len(c) for c in sentence_chunks) / len(sentence_chunks):.0f} chars")
print()
print("=== Sample chunk (middle of document) ===")
print(sentence_chunks[len(sentence_chunks) // 2])

Number of chunks: 117
Average chunk length: 1172 chars

=== Sample chunk (middle of document) ===
The
burden of HTN on those populations can be considerable (82). There are very little data on HTN control,
access to care and treatment, and patient understanding of HTN from Africa and Asia (except Japan),
despite protracted refugee situations on these continents. Violent and protracted conflicts are disastrous
to civilian populations and their health care systems, and result in interruptions to treatment and care
(83, 84). Armed conflicts are associated with increased short-term and long-term cardiac morbidity and
mortality and increases in blood pressure (BP) (85). Following exposure to conflict, research in military
populations shows that post-traumatic stress disorder and severe injury are independent risk factors for
the development of HTN (86). The rates of treatment ranged from 53.4% to 98.1% of patients with
HTN in this population (87, 88). There are currently no data regarding t

## Technique 3 — Recursive/Structure-Aware Chunking

In [ ]:
import re

def recursive_chunk(text: str, target_tokens: int = 300) -> list:
    """
    Split text primarily by paragraph breaks; only fall back to
    sentence-splitting when a single paragraph exceeds target_tokens.
    """
    paragraphs = re.split(r'\n\s*\n', text)  # split on blank-line paragraph breaks
    chunks = []
    current_chunk = []
    current_tokens = 0

    def flush():
        if current_chunk:
            chunks.append("\n\n".join(current_chunk))

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
        para_tokens = len(encoding.encode(para))

        if para_tokens > target_tokens:
            # This single paragraph is too big - fall back to sentence-splitting just for it
            flush()
            current_chunk = []
            current_tokens = 0
            sub_chunks = sentence_based_chunk(para, target_tokens=target_tokens)
            chunks.extend(sub_chunks)
            continue

        if current_tokens + para_tokens > target_tokens and current_chunk:
            flush()
            current_chunk = []
            current_tokens = 0

        current_chunk.append(para)
        current_tokens += para_tokens

    flush()
    return chunks


recursive_chunks = recursive_chunk(who_sample.clean_text, target_tokens=300)

print(f"Number of chunks: {len(recursive_chunks)}")
print(f"Average chunk length: {sum(len(c) for c in recursive_chunks) / len(recursive_chunks):.0f} chars")
print()
print("=== Sample chunk (middle of document) ===")
print(recursive_chunks[len(recursive_chunks) // 2])

Number of chunks: 117
Average chunk length: 1172 chars

=== Sample chunk (middle of document) ===
The
burden of HTN on those populations can be considerable (82). There are very little data on HTN control,
access to care and treatment, and patient understanding of HTN from Africa and Asia (except Japan),
despite protracted refugee situations on these continents. Violent and protracted conflicts are disastrous
to civilian populations and their health care systems, and result in interruptions to treatment and care
(83, 84). Armed conflicts are associated with increased short-term and long-term cardiac morbidity and
mortality and increases in blood pressure (BP) (85). Following exposure to conflict, research in military
populations shows that post-traumatic stress disorder and severe injury are independent risk factors for
the development of HTN (86). The rates of treatment ranged from 53.4% to 98.1% of patients with
HTN in this population (87, 88). There are currently no data regarding t

## Technique 4 — Sliding Window Chunking

In [34]:
def sliding_window_chunk(text: str, target_tokens: int = 300, overlap_sentences: int = 2) -> list:
    """
    Split text into sentence-respecting chunks, with the last
    overlap_sentences of each chunk repeated as the start of the next.
    """
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)

    chunks = []
    current_chunk = []
    current_tokens = 0

    i = 0
    while i < len(sentences):
        sentence = sentences[i]
        sentence_tokens = len(encoding.encode(sentence))

        if current_tokens + sentence_tokens > target_tokens and current_chunk:
            chunks.append(" ".join(current_chunk))
            # Start next chunk with the last N sentences of this one (overlap)
            current_chunk = current_chunk[-overlap_sentences:] if len(current_chunk) >= overlap_sentences else current_chunk[:]
            current_tokens = sum(len(encoding.encode(s)) for s in current_chunk)

        current_chunk.append(sentence)
        current_tokens += sentence_tokens
        i += 1

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


sliding_chunks = sliding_window_chunk(who_sample.clean_text, target_tokens=300, overlap_sentences=2)

print(f"Number of chunks: {len(sliding_chunks)}")
print(f"Average chunk length: {sum(len(c) for c in sliding_chunks) / len(sliding_chunks):.0f} chars")
print()
mid = len(sliding_chunks) // 2
print("=== Two Consecutive Chunks (showing the overlap) ===")
print("--- Chunk N ---")
print(sliding_chunks[mid][-600:])  # end of one chunk
print()
print("--- Chunk N+1 ---")
print(sliding_chunks[mid + 1][:600])  # start of next chunk

Number of chunks: 174
Average chunk length: 1307 chars

=== Two Consecutive Chunks (showing the overlap) ===
--- Chunk N ---
tes and chronic kidney disease. Concerns regarding use of angiotensin-converting enzyme inhibitors (ACEis) in these patients were
raised due to identification of angiotensin-converting enzyme 2 (ACE2), the monocarboxypeptidase
that inactivates angiotensin II and thereby counters the activation of the classic renin–angiotensin–
aldosterone system (RAAS), as the functional receptor for the severe acute respiratory syndrome
coronavirus 2 (SARS-CoV-2) (95, 96). The WHO conducted a rapid review of evidence related to the use
ACEis or ARBs in COVID patients which identified 11 observational studies.

--- Chunk N+1 ---
Concerns regarding use of angiotensin-converting enzyme inhibitors (ACEis) in these patients were
raised due to identification of angiotensin-converting enzyme 2 (ACE2), the monocarboxypeptidase
that inactivates angiotensin II and thereby counters the act

## Technique 5 — Semantic Chunking --> Embed Each Sentence

In [9]:
from openai import OpenAI
import numpy as np

client = OpenAI()  # reads OPENAI_API_KEY from environment/.env automatically

sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', who_sample.clean_text)
print(f"Total sentences: {len(sentences)}")

# Embed in batches to avoid one huge request
def get_embeddings(texts, batch_size=100):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(model="text-embedding-3-small", input=batch)
        all_embeddings.extend([e.embedding for e in response.data])
    return np.array(all_embeddings)

sentence_embeddings = get_embeddings(sentences)
print("Embeddings shape:", sentence_embeddings.shape)

Total sentences: 743
Embeddings shape: (743, 1536)


## Detect Semantic Boundaries (Similarity Drops) and Build Chunks

In [10]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def semantic_chunk(sentences: list, embeddings: np.ndarray, similarity_threshold: float = 0.5, max_tokens: int = 400) -> list:
    """
    Group sentences into chunks, starting a new chunk wherever consecutive
    sentence similarity drops below similarity_threshold (topic shift),
    or when max_tokens would be exceeded (safety cap).
    """
    chunks = []
    current_chunk = [sentences[0]]
    current_tokens = len(encoding.encode(sentences[0]))

    for i in range(1, len(sentences)):
        sim = cosine_similarity(embeddings[i - 1], embeddings[i])
        sentence_tokens = len(encoding.encode(sentences[i]))

        if (sim < similarity_threshold or current_tokens + sentence_tokens > max_tokens) and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_tokens = 0

        current_chunk.append(sentences[i])
        current_tokens += sentence_tokens

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


semantic_chunks = semantic_chunk(sentences, sentence_embeddings, similarity_threshold=0.5, max_tokens=400)

print(f"Number of chunks: {len(semantic_chunks)}")
print(f"Average chunk length: {sum(len(c) for c in semantic_chunks) / len(semantic_chunks):.0f} chars")
print()
print("=== Sample chunk (middle of document) ===")
print(semantic_chunks[len(semantic_chunks) // 2])

Number of chunks: 474
Average chunk length: 289 chars

=== Sample chunk (middle of document) ===
Humanitarian crises and disaster settings (natural or humanmade) can affect health care and services
in many different ways.


## Retry Semantic Chunking With a Lower Threshold

In [11]:
semantic_chunks_v2 = semantic_chunk(sentences, sentence_embeddings, similarity_threshold=0.3, max_tokens=400)

print(f"Number of chunks: {len(semantic_chunks_v2)}")
print(f"Average chunk length: {sum(len(c) for c in semantic_chunks_v2) / len(semantic_chunks_v2):.0f} chars")
print()
print("=== Sample chunk (middle of document) ===")
print(semantic_chunks_v2[len(semantic_chunks_v2) // 2])

Number of chunks: 128
Average chunk length: 1071 chars

=== Sample chunk (middle of document) ===
Presumably, health
inequities are reduced, since task shifting in the public sector increases access to those using public
health vs private health. Increasing access in underserved areas can improve inequities.


## Technique 6 — Upgrade Sentence Splitting to spaCy

In [35]:
import spacy

nlp = spacy.load("en_core_web_sm")


def spacy_sentence_split(text: str) -> list:
    """Split text into sentences using spaCy's trained sentence boundary detector,
    which correctly handles medical abbreviations (Dr., e.g., mg., Fig., vs.)
    unlike a naive regex splitter."""
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]


def sentence_based_chunk_v2(text: str, target_tokens: int = 300) -> list:
    """Same grouping logic as before, now using spaCy for sentence detection."""
    sentences = spacy_sentence_split(text)

    chunks = []
    current_chunk = []
    current_tokens = 0

    for sentence in sentences:
        sentence_tokens = len(encoding.encode(sentence))

        if current_tokens + sentence_tokens > target_tokens and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_tokens = 0

        current_chunk.append(sentence)
        current_tokens += sentence_tokens

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


spacy_chunks = sentence_based_chunk_v2(who_sample.clean_text, target_tokens=300)

print(f"Number of chunks: {len(spacy_chunks)}")
print(f"Average chunk length: {sum(len(c) for c in spacy_chunks) / len(spacy_chunks):.0f} chars")
print()

# Check specifically for an abbreviation-related sentence to confirm correct splitting
for i, sent in enumerate(spacy_sentence_split(who_sample.clean_text)):
    if re.search(r'\b(e\.g\.|i\.e\.|vs\.|Dr\.|Fig\.)\b', sent) and len(sent) < 300:
        print(f"Sentence {i}: {sent}")
        break

Number of chunks: 118
Average chunk length: 1162 chars



## Direct Before/After Comparison on an Abbreviation

In [36]:
# Find a real sentence in the WHO text containing a common medical abbreviation
test_snippet = "Patients with HTN (e.g. those over 65 years) should be monitored closely. Treatment options vs. placebo were compared in Fig. 3 of the study."

# OLD regex-based splitter
old_split = re.split(r'(?<=[.!?])\s+(?=[A-Z])', test_snippet)

# NEW spaCy-based splitter
new_split = spacy_sentence_split(test_snippet)

print("=== OLD (regex) split ===")
for s in old_split:
    print(repr(s))

print()
print("=== NEW (spaCy) split ===")
for s in new_split:
    print(repr(s))

=== OLD (regex) split ===
'Patients with HTN (e.g. those over 65 years) should be monitored closely.'
'Treatment options vs. placebo were compared in Fig. 3 of the study.'

=== NEW (spaCy) split ===
'Patients with HTN (e.g. those over 65 years) should be monitored closely.'
'Treatment options vs. placebo were compared in Fig.'
'3 of the study.'


## Add Custom Abbreviation Exceptions to spaCy

In [37]:
import spacy
from spacy.language import Language

nlp = spacy.load("en_core_web_sm", exclude=["parser"])
nlp.add_pipe("sentencizer")

ABBREVIATIONS = {
    "Fig", "Eq", "Ref", "Vol", "e.g.", "i.e.", "vs.", "Dr", "Mr", "Mrs",
    "et al", "approx", "cf", "mg", "mL", "no", "cm", "kg",
}

if "fix_abbreviation_boundaries" in nlp.pipe_names:
    nlp.remove_pipe("fix_abbreviation_boundaries")

@Language.component("fix_abbreviation_boundaries")
def fix_abbreviation_boundaries(doc):
    for i, token in enumerate(doc[:-1]):
        text = token.text.rstrip(".")
        is_abbrev = text in ABBREVIATIONS or token.text in ABBREVIATIONS

        if is_abbrev:
            # If the abbreviation's period is its own separate token
            # (e.g. "Fig" + "."), the boundary lands on the token AFTER
            # that period - so unset it two tokens ahead, not one.
            if token.text != "." and i + 1 < len(doc) and doc[i + 1].text == "." and i + 2 < len(doc):
                doc[i + 2].is_sent_start = False
            elif i + 1 < len(doc):
                doc[i + 1].is_sent_start = False
    return doc

nlp.add_pipe("fix_abbreviation_boundaries", after="sentencizer")

new_split_fixed = spacy_sentence_split(test_snippet)
print("=== FIXED spaCy split ===")
for s in new_split_fixed:
    print(repr(s))

=== FIXED spaCy split ===
'Patients with HTN (e.g. those over 65 years) should be monitored closely.'
'Treatment options vs. placebo were compared in Fig. 3 of the study.'


## Re-run Full Chunking on WHO Sample With the Fixed Splitter

In [25]:
def sentence_based_chunk_final(text: str, target_tokens: int = 300) -> list:
    """Group spaCy-detected sentences (with abbreviation fix) into target-sized chunks."""
    sentences = spacy_sentence_split(text)

    chunks = []
    current_chunk = []
    current_tokens = 0

    for sentence in sentences:
        sentence_tokens = len(encoding.encode(sentence))

        if current_tokens + sentence_tokens > target_tokens and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_tokens = 0

        current_chunk.append(sentence)
        current_tokens += sentence_tokens

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


final_chunks = sentence_based_chunk_final(who_sample.clean_text, target_tokens=300)

print(f"Number of chunks: {len(final_chunks)}")
print(f"Average chunk length: {sum(len(c) for c in final_chunks) / len(final_chunks):.0f} chars")

Number of chunks: 113
Average chunk length: 1214 chars


## Technique 7 — Detect Numbered Section Headers in the WHO Text

In [26]:
import re

# Common pattern for clinical guideline section headers: "5.1 Hypoglycemia",
# "2.1 Recommended Dosing", "4.2.1 Some Subsection", etc.
SECTION_HEADER_PATTERN = re.compile(r'\n(\d+(?:\.\d+){0,3}\s+[A-Z][A-Za-z][^\n]{2,80})\n')

matches = SECTION_HEADER_PATTERN.findall(who_sample.clean_text)

print(f"Number of section headers detected: {len(matches)}")
print()
print("=== Sample detected headers ===")
for m in matches[:20]:
    print(repr(m))

Number of section headers detected: 82

=== Sample detected headers ===
'1 Introduction 1'
'2.1 Guideline contributors 3'
'2.3 Outcome importance rating 4'
'2.5 Certainty of evidence and strength of recommendations 5'
'2.7 Funding 6'
'3.1 Blood pressure threshold for initiation of pharmacological treatment 7'
'3.3 Cardiovascular disease risk assessment as guide to initiation of'
'3.4 Drug classes to be used as first-line agents 11'
'3.6 Target blood pressure 16'
'3.8 Administration of treatment by nonphysician professionals 19'
'4.1 Hypertension in disaster, humanitarian and emergency settings 21'
'4.3 Pregnancy and hypertension 22'
'5.1 Publication 24'
'5.3 Evaluation 24'
'5.5 Research gaps 24'
'6.1 Guideline recommendations 26'
'1 Introduction'
'2 GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS'
'2.1 Guideline contributors'
'2.2 Analytical framework and PICOs'


## Detect Where the Body Starts (Skip the TOC)

In [27]:
# Find where headers start repeating - that's where the real body begins
seen_headers = set()
body_start_pos = None

for match in SECTION_HEADER_PATTERN.finditer(who_sample.clean_text):
    header_text = match.group(1)
    normalized = re.sub(r'\s+\d+$', '', header_text).strip()  # strip trailing page number for comparison

    if normalized in seen_headers:
        body_start_pos = match.start()
        print(f"Body likely starts at character {body_start_pos}, header: {repr(header_text)}")
        break

    seen_headers.add(normalized)

print(f"\nTotal document length: {len(who_sample.clean_text)}")
print(f"TOC portion: ~{body_start_pos} chars ({body_start_pos / len(who_sample.clean_text) * 100:.1f}% of document)")

Body likely starts at character 12166, header: '1 Introduction'

Total document length: 137278
TOC portion: ~12166 chars (8.9% of document)


## Section-Header-Based Chunking

In [28]:
def section_based_chunk(text: str, header_pattern=SECTION_HEADER_PATTERN, max_tokens: int = 600) -> list:
    """
    Split text by the document's own numbered section headers (e.g. "5.1
    Hypoglycemia"), skipping the Table of Contents. Each section becomes one
    chunk if it fits within max_tokens; oversized sections fall back to
    spaCy sentence-grouping internally.
    """
    # Find where the real body starts (skip TOC)
    seen_headers = set()
    body_start = 0
    header_positions = []

    for match in header_pattern.finditer(text):
        header_text = match.group(1)
        normalized = re.sub(r'\s+\d+$', '', header_text).strip()

        if normalized in seen_headers and body_start == 0:
            body_start = match.start()

        if match.start() >= body_start and body_start > 0:
            header_positions.append(match.start())

        seen_headers.add(normalized)

    if not header_positions:
        # No real body headers found - fall back entirely to sentence chunking
        return sentence_based_chunk_final(text, target_tokens=300)

    # Split the body text at each header position
    body_text = text[body_start:]
    adjusted_positions = [p - body_start for p in header_positions if p >= body_start]

    sections = []
    for i, pos in enumerate(adjusted_positions):
        end = adjusted_positions[i + 1] if i + 1 < len(adjusted_positions) else len(body_text)
        sections.append(body_text[pos:end])

    # For each section, keep as one chunk if small enough, else sub-split with spaCy
    chunks = []
    for section in sections:
        section_tokens = len(encoding.encode(section))
        if section_tokens <= max_tokens:
            chunks.append(section.strip())
        else:
            chunks.extend(sentence_based_chunk_final(section, target_tokens=300))

    return chunks


section_chunks = section_based_chunk(who_sample.clean_text)

print(f"Number of chunks: {len(section_chunks)}")
print(f"Average chunk length: {sum(len(c) for c in section_chunks) / len(section_chunks):.0f} chars")
print()
print("=== Sample chunk (a real section) ===")
print(section_chunks[5][:600])

Number of chunks: 124
Average chunk length: 1008 chars

=== Sample chunk (a real section) ===
2 GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS
2 Method for developing the
guideline


## Inspect a Content-Rich Section

In [29]:
# Look for a section with real substance - print a few candidates by length
section_lengths = [(i, len(c)) for i, c in enumerate(section_chunks)]
section_lengths.sort(key=lambda x: -x[1])  # longest first

print("Longest sections (likely to contain real recommendation + rationale content):")
for i, length in section_lengths[:5]:
    print(f"  Chunk {i}: {length} chars")

print()
print("=== Full content of the longest section ===")
print(section_chunks[section_lengths[0][0]])

Longest sections (likely to contain real recommendation + rationale content):
  Chunk 9: 2928 chars
  Chunk 66: 2689 chars
  Chunk 54: 2666 chars
  Chunk 21: 2644 chars
  Chunk 27: 2636 chars

=== Full content of the longest section ===
2.4 Reviews of evidence
The WHO Steering Group, with the participation of the GDG, determined the scope of the guideline
and identified eleven questions in the format of population, intervention, comparison, and outcomes
(PICO) to guide the search for systematic reviews (Annex 4). Eleven overviews of reviews informed the
guideline development process. A systematic search was carried out in PubMed, Embase, The Cochrane
Library, and Epistemonikos to identify existing systematic reviews that answered the PICO questions
published in 2015 or after. Suitable systematic reviews were then evaluated based on the following
criteria:
[ their methodology as appraised by the AMSTAR (Assessing the Methodological Quality of
Systematic Reviews) tool;
[ how directly the

## Test Section Detection on a Second, Different Document

In [30]:
malaria_sample = load_guideline("malaria", output_dir="../data/raw/who")

malaria_matches = SECTION_HEADER_PATTERN.findall(malaria_sample.clean_text)
print(f"Malaria: {len(malaria_matches)} potential headers detected")
print("Sample:", malaria_matches[:10])

print()

mhgap_sample = load_guideline("depression", output_dir="../data/raw/who")  # mhGAP document
mhgap_matches = SECTION_HEADER_PATTERN.findall(mhgap_sample.clean_text)
print(f"mhGAP (depression): {len(mhgap_matches)} potential headers detected")
print("Sample:", mhgap_matches[:10])

Malaria: 94 potential headers detected
Sample: ['2023.01 Rev.1). License: CC BY-NC-SA 3.0 IGO.', '2.1 Guideline translations', '4.1 Vector control', '4.1.2 Co-deploying ITNs and IRS', '4.1.3 Supplementary interventions', '4.1.4 Research needs', '4.2.1 Intermittent preventive treatment of malaria in pregnancy (IPTp)', '4.2.3 Seasonal malaria chemoprevention (SMC)', '4.2.4 Intermittent preventive treatment of malaria in school-aged children (IPTsc)', '4.2.5 Post-discharge malaria chemoprevention (PDMC)']

mhGAP (depression): 2 potential headers detected
Sample: ['8\nThe GRADE methodology can be found at http://www.gradeworkinggroup.org/index.htm', '55\nAppendix 3: Overview of declarations of interest from GDG']


## Check Section-Header Detection Across More Documents

In [31]:
topics_to_check = ["tuberculosis", "covid-19", "breast_cancer", "typhoid", "anemia_in_pregnancy", "malnutrition"]

for topic in topics_to_check:
    sample = load_guideline(topic, output_dir="../data/raw/who")
    if sample is None:
        print(f"{topic}: no guideline found")
        continue
    matches = SECTION_HEADER_PATTERN.findall(sample.clean_text)
    print(f"{topic}: {len(matches)} potential headers")
    print(f"  Sample: {matches[:3]}")
    print()

tuberculosis: 1 potential headers
  Sample: ['6 See: https://www.who.int/health-topics/tuberculosis']

covid-19: 0 potential headers
  Sample: []

breast_cancer: 88 potential headers
  Sample: ['20 Avenue Appia', '4\nAcknowledgements', '5\nWHO position paper on mammography screening']

typhoid: 47 potential headers
  Sample: ['1.1 The organism', '1.2 The disease', '1.2.1 Symptoms']

anemia_in_pregnancy: 37 potential headers
  Sample: ['2 Guideline on haemoglobin cutoffs to define anaemia in individuals and populations', '3\nExisting WHO documents related to this new guideline', '4 Guideline on haemoglobin cutoffs to define anaemia in individuals and populations']

malnutrition: 14 potential headers
  Sample: ['6.2 Full-strength, standard WHO low-osmolarity oral rehydration solution (75 mmol/L of', '6.3 ReSoMal2 (or locally prepared ReSoMal using standard WHO low-osmolarity oral', '1 Three or more loose or watery stools in a day, for more than 14 days.']



# Final Chunking Technique is 	Sentence-based chunking, spaCy + abbreviation fix

## Final Review Fix: Deduplicating Shared WHO Documents

A final end-to-end review of this phase (before moving to Phase 7) found that
topics sharing one underlying WHO document (asthma/copd, the 4-way CVD group,
depression/anxiety/epilepsy) were each being chunked independently, producing
~24% exact content duplicates (6,336 chunks where only 4,796 were unique).
Also found: `chunk_id` strings aren't valid Qdrant point IDs (Qdrant requires
an unsigned int or UUID).

Fixed by: changing `Chunk.topic` to `Chunk.topics: List[str]`, adding a
deterministic `point_id`, and chunking each shared document once using a
canonical id (topics joined with "+") instead of once per topic.

In [2]:
# Confirm point_id determinism
id1 = Chunk.make_point_id("asthma+copd_who_text_0")
id2 = Chunk.make_point_id("asthma+copd_who_text_0")
print("Same chunk_id -> same point_id:", id1 == id2)
print("Example point_id:", id1)

# Canonical id used for a document shared across topics
shared_topics = ["copd", "asthma"]
canonical_id = "+".join(sorted(shared_topics))
print("Canonical id for a shared document:", canonical_id)

Same chunk_id -> same point_id: True
Example point_id: c7cc4fb0-da32-56df-96ee-2b079025335b
Canonical id for a shared document: asthma+copd


In [3]:
# Final production run results (from run_chunking.py):
print("PubMed:  5,391 chunks")
print("OpenFDA: 18,068 chunks")
print("WHO:     4,796 unique chunks (down from 6,336 before the dedup fix)")
print("Total:   28,255 chunks")

PubMed:  5,391 chunks
OpenFDA: 18,068 chunks
WHO:     4,796 unique chunks (down from 6,336 before the dedup fix)
Total:   28,255 chunks
